# 03 · U-Net
Bar to beat: the random forest (B2), IoU 0.704 on unseen sites. The U-Net works on whole image patches rather than single pixels, so it can use shape and context. It uses the same 5 site folds, labels and scoring code as the baselines, so the numbers are directly comparable. The locked Test_Dataset is never opened.

## Environment
Runs in the `mangrove` conda env (setup in `RUN_LOCALLY.md`).

## Paths
`paths.py` defines every folder in one place; Jupyter is launched from the SDP folder, so no paths need editing.
* `3_Data/`: small inputs (sites, labels, image list, folds)
* `6_Work/`: everything the pipeline rebuilds (images, chips, datasets), about 4 GB
* `4_Results/`: tables and figures

In [ ]:
import os, subprocess, sys
from pathlib import Path

for _d in [Path.cwd(), *Path.cwd().parents]:          # works from SDP, SDP/2_Code or SDP/2_Code/notebooks
    if (_d / 'paths.py').exists():
        CODE = _d; break
    if (_d / '2_Code' / 'paths.py').exists():
        CODE = _d / '2_Code'; break
else:
    raise FileNotFoundError('2_Code/paths.py not found: start Jupyter inside the SDP folder')
sys.path.insert(0, str(CODE))
import paths as P

PROJECT_ROOT, LAYOUT = P.PROJECT_ROOT, 'sdp'
WORK, SITES, LABELS, SCENES, FOLDS, QA_CSV = P.WORK, P.SITES, P.LABELS, P.SCENES, P.FOLDS, P.QA_CSV
DATASETS, RESULTS = P.DATASETS, P.RESULTS
TRAINVAL = P.TRAIN_VAL            # the Test_Dataset folder is never opened in these notebooks

def run_step(*args):
    """Run 2_Code/run_all.py with these arguments and show its output here, live."""
    env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([sys.executable, str(CODE / 'run_all.py'), *map(str, args)], cwd=PROJECT_ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError('run_all.py failed: see the messages above')

print(P.describe())

In [ ]:
import json, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 11})
pd.set_option('display.width', 160)

## Settings
* `RUN_TRAINING = False` only shows the saved results; `True` retrains.
* `PRESET` selects the model configuration defined in `train_unet.py` (base channels, crop size, epochs, patience).

In [ ]:
RUN_TRAINING = True
PRESET = 'gpu'

import train_unet as TU
CONFIG_PRESETS = TU.PRESETS
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', DEVICE, '| preset:', PRESET, CONFIG_PRESETS[PRESET])

UNET_DIR = P.unet_dir(PRESET)
BASE_DIR = P.BASELINES
print('U-Net results folder:', UNET_DIR)
print('Baseline results folder:', BASE_DIR)

## Step 1 · Architecture
The encoder (left side of the U) shrinks the 256 × 256 input step by step to capture what is present; the decoder (right side) expands it back to full size to place it, with skip connections keeping the fine edges. The output is a mangrove probability per pixel, thresholded into a yes/no map.

In [ ]:
model = TU.UNet(8, CONFIG_PRESETS[PRESET]['base'])
n_params = sum(p.numel() for p in model.parameters())
print(f'U-Net with 8 input channels (B02 B03 B04 B08 B11 B12 NDVI MNDWI), {n_params:,} parameters')
levels = [8] + [CONFIG_PRESETS[PRESET]['base'] * m for m in (1, 2, 4, 8, 16)]
fig, ax = plt.subplots(figsize=(8, 2.6))
for i, c in enumerate(levels[1:]):
    h = 1.6 - 0.28 * i
    ax.add_patch(plt.Rectangle((i * 0.9, -h / 2), 0.6, h, color='#2f6aa3' if i < 4 else '#555'))
    ax.text(i * 0.9 + 0.3, h / 2 + 0.08, f'{c} ch', ha='center', fontsize=8)
    if i < 4:
        j = 8 - i
        ax.add_patch(plt.Rectangle((j * 0.9, -h / 2), 0.6, h, color='#2e8b57'))
        ax.annotate('', xy=(j * 0.9, 0), xytext=(i * 0.9 + 0.6, 0), arrowprops=dict(arrowstyle='->', color='#999', lw=0.8))
ax.text(0.3, -1.05, 'encoder: what is there', fontsize=9); ax.text(6.2, -1.05, 'decoder: where it is', fontsize=9)
ax.set_xlim(-0.2, 8.4); ax.set_ylim(-1.2, 1.2); ax.axis('off'); ax.set_title('U-Net shape (grey arrows = skip connections)')
plt.show()

## Step 2 · Folds
Same folds as the baselines, one Abu Dhabi site held out per fold. The U-Net trains on the rest (hazy scenes dropped, same rule as the baselines) with one further fold kept as inner validation, which sets the stopping epoch and the threshold. The held-out site is predicted only at the end.

In [ ]:
fits_path = UNET_DIR / 'fold_fits.csv'
if fits_path.exists():
    fits = pd.read_csv(fits_path)
    display(fits[['fold', 'held_site', 'held_chips', 'train_chips', 'inner_fold', 'inner_chips', 'best_epoch', 'epochs_run', 'threshold', 'minutes']])
else:
    print('No fold_fits.csv yet: run the training first (RUN_TRAINING = True).')

## Step 3 · Training (only if `RUN_TRAINING = True`)
Runs all 5 folds and prints the loss and inner-validation IoU every epoch. Weights and predictions go to `6_Work/runs`, tables and figures to `4_Results`.

In [ ]:
if RUN_TRAINING:
    run_step('--step', 'unet', 'score_unet', '--preset', PRESET)   # = python 2_Code/run_all.py --step unet score_unet --preset ...
else:
    print('Skipping training: showing the saved results from', UNET_DIR)

## Step 4 · Training curves
Left: training loss. Right: inner-validation IoU, with a dot at the epoch whose weights were kept (early stopping).

In [ ]:
hist_path = UNET_DIR / 'history.csv'
if hist_path.exists():
    hist = pd.read_csv(hist_path); fits = pd.read_csv(UNET_DIR / 'fold_fits.csv')
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    for k, g in hist.groupby('fold'):
        site = fits.query('fold==@k').held_site.iloc[0].replace('_', ' ')
        axs[0].plot(g.epoch, g.train_loss, label=f'F{k} {site[:18]}')
        axs[1].plot(g.epoch, g['inner_val_iou@0.5'], label=f'F{k}')
        be = fits.query('fold==@k').best_epoch.iloc[0]
        axs[1].scatter([be], [g.query('epoch==@be')['inner_val_iou@0.5'].iloc[0]], s=25)
    axs[0].set_title('Training loss (BCE + Dice)'); axs[0].set_xlabel('epoch'); axs[0].legend(fontsize=8)
    axs[1].set_title('Inner-validation IoU (threshold 0.5)'); axs[1].set_xlabel('epoch'); axs[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()
else:
    print('RESULTS PENDING: no history.csv yet.')

## Step 5 · Comparison with the baselines
IoU on each held-out site with 95% confidence intervals, scored with the same code as the baselines. Dashed line: B2 = 0.704.

In [ ]:
cv_path = UNET_DIR / 'cv_results.csv'
if cv_path.exists():
    u = pd.read_csv(cv_path).set_index('fold')
    base = pd.read_csv(BASE_DIR / 'cv_results.csv').query("variant=='b_no_haze'")
    b1 = base.query("model=='B1'").set_index('fold'); b2 = base.query("model=='B2'").set_index('fold')
    summary = pd.DataFrame({m: {k: f"{d[k].mean():.3f} ± {d[k].std():.3f}" for k in ['iou', 'f1', 'precision', 'recall']}
                            for m, d in [('U-Net', u), ('B2 random forest', b2), ('B1 NDVI threshold', b1)]}).T
    display(summary)
    fig, ax = plt.subplots(figsize=(10, 4)); x = np.arange(len(u)); w = 0.27
    for i, (lab, d, col) in enumerate([('B1 NDVI threshold', b1, '#2f6aa3'), ('B2 random forest', b2, '#c9571c'), ('U-Net', u, '#2e8b57')]):
        v = d.loc[u.index, 'iou'].values
        ax.bar(x + (i - 1) * w, v, w, color=col, label=lab,
               yerr=[v - d.loc[u.index, 'iou_ci95_lo'].values, d.loc[u.index, 'iou_ci95_hi'].values - v], capsize=2)
    ax.axhline(b2.loc[u.index, 'iou'].mean(), color='#c9571c', ls='--', lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels([f"F{k}\n{u.loc[k, 'held_site'].replace('_', ' ')[:22]}" for k in u.index], fontsize=8)
    ax.set_ylim(0, 1); ax.set_ylabel('IoU on held-out site'); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
    diff = (u.iou - b2.loc[u.index, 'iou']).round(3)
    print('U-Net minus B2 per fold:', dict(diff)); print('U-Net wins on', int((diff > 0).sum()), 'of 5 folds')
else:
    print('RESULTS PENDING: no cv_results.csv yet.')

## Step 6 · Example predictions
The same three chips as the baselines notebook: image, GMW label, random forest, U-Net.

In [ ]:
from IPython.display import Image, display as _d
ex = UNET_DIR / 'figs' / 'examples.png'
if ex.exists():
    _d(Image(filename=str(ex)))
else:
    print('RESULTS PENDING: examples.png not found.')

## Step 7 · Written summary
`RESULTS.md`, with the per-fold table.

In [ ]:
from IPython.display import Markdown
res_md = UNET_DIR / 'RESULTS.md'
_d(Markdown(res_md.read_text(encoding='utf-8'))) if res_md.exists() else print('RESULTS PENDING: RESULTS.md not written yet.')